# MS4215 · Python Training 01
## What does data analysis actually do? — Irish weather

**College order:** Week 1A → Week 1B, before the numbered labs.  
**Original lecturer/material:** Associate Professor Sinéad Moylett, University of Limerick.  
**This notebook:** an independent Python teaching adaptation of the supplied Week 1 material.

> **Big idea:** Data analysis is a method for turning observations into a defensible answer to a question. Code is a tool; the statistical reasoning is the subject.

**Learning route:** Question → data → quality → visualisation → summary → interpretation → limitations.

**How to work:** Run one cell, predict the next output, write an answer in your own words, and only then move on. In Colab choose **Runtime → Run all**, or run one cell at a time using Shift+Enter.

**Dataset:** the original Irish weather CSV (2000–2023), already on Erik's website. The notebook also accepts a local copy in the current or adjacent data directory.


## 1 · Ask a question before touching the data

**Investigation:** Which weather stations typically record more daily rainfall?

That simple question hides important decisions:
- Do we mean the **average** rainfall or rainfall on a **typical** observed day?
- What should happen to **missing observations**?
- Are we comparing **all observed dates**, or like-for-like days?
- Could any pattern tell us the **cause** of rainfall? (Not from these observations alone.)

**Write first:** What difference would you expect between the mean and median of rainfall, and why?


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def load_course_csv(filename):
    """Prefer the files in a local course folder; otherwise use the published course copy."""
    for candidate in [Path("data") / filename, Path("../data") / filename, Path(filename)]:
        if candidate.is_file():
            print("Reading local file:", candidate)
            return pd.read_csv(candidate, na_values=["NA"])
    url = ("https://raw.githubusercontent.com/ErikDownes/"
           "ErikDownes.github.io/main/resources/ms4215/data/" + filename)
    print("Reading course dataset from GitHub")
    return pd.read_csv(url, na_values=["NA"])

weather = load_course_csv("ireland_weather_2000_2023.csv")
weather["date"] = pd.to_datetime(weather["date"], errors="coerce")
print("Rows, columns:", weather.shape)
display(weather.head(8))


## 2 · Understand what one row and one column mean

One **row** represents a station/date observation. One **column** represents a variable:
- station: categorical (nominal)
- date: a time coordinate
- tmax_c, tmin_c: numerical temperature measurements in Celsius
- prcp_mm: daily precipitation, measured in millimetres

**Important distinction:** Zero rainfall means no recorded rainfall. Missing rainfall means *we don't know*. Never silently convert missing values to zero.

*Statistical precision:* Celsius is an **interval** temperature scale, not a ratio scale: 20°C is not "twice as hot" as 10°C.


In [ ]:
print("Data types:")
print(weather.dtypes.to_string())
print("\nNumber of observations at each station:")
print(weather["station"].value_counts().to_string())
print("\nOne observation per station-date?")
print("Duplicate station-date rows:", weather.duplicated(["station", "date"]).sum())


## 3 · Data quality is analysis, not housekeeping

An analyst doesn't simply remove every inconvenient row. First measure the problem.

**Think:** If some stations have more missing precipitation measurements, can we confidently compare their observed distributions?


In [ ]:
missing = pd.DataFrame({
    "missing_rows": weather.isna().sum(),
    "percent_missing": 100 * weather.isna().mean()
}).round(1)
display(missing)

missing_by_station = (
    weather.groupby("station")["prcp_mm"]
    .apply(lambda s: 100 * s.isna().mean())
    .sort_values(ascending=False)
)
display(missing_by_station.rename("precipitation missing (%)").round(1).to_frame())

print("Rows retained for rainfall analysis:", weather["prcp_mm"].notna().sum())


## 4 · First look at the shape of the data

Histograms answer: *What kinds of values occur, and how often?*

Look for symmetry, skewness, clusters, unusual values and concentrations at zero. **Don't pick the summary statistic before looking at the distribution.**


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
weather["tmax_c"].dropna().hist(bins=35, ax=axes[0])
axes[0].set(title="Daily maximum temperature", xlabel="°C", ylabel="Observed days")
weather["prcp_mm"].dropna().hist(bins=45, ax=axes[1])
axes[1].set(title="Daily precipitation", xlabel="mm", ylabel="Observed days")
plt.tight_layout()
plt.show()


## 5 · Choose statistics that answer the question

For a reasonably symmetric distribution, a **mean** and **standard deviation** often communicate centre and spread well.

For a heavily right-skewed distribution, a **median** and **interquartile range (IQR)** are often more representative of a typical observation.

Recall: IQR = 75th percentile − 25th percentile. The mean is sensitive to very large values; the median is more resistant.


In [ ]:
def describe_distribution(series):
    s = series.dropna()
    return pd.Series({
        "observations": len(s),
        "mean": s.mean(),
        "median": s.median(),
        "SD": s.std(),
        "Q1": s.quantile(.25),
        "Q3": s.quantile(.75),
        "IQR": s.quantile(.75) - s.quantile(.25)
    }).round(2)

display(pd.DataFrame({
    "Max temperature": describe_distribution(weather["tmax_c"]),
    "Rainfall": describe_distribution(weather["prcp_mm"])
}))
print("Reminder: the summaries exclude missing measurements, not by converting them to zero.")


## 6 · Compare groups fairly

We can group observations by station, but remember:
- The median includes genuinely **dry days** (zero rainfall).
- A missing value is excluded, not treated as a dry day.
- Unequal observation coverage may distort comparisons.
- Group differences are **associations**, not causal explanations.

**Before running:** Which station do you expect to have the highest median daily precipitation?


In [ ]:
station_summary = (
    weather.groupby("station")
    .agg(
        observed_rainfall_days=("prcp_mm", "count"),
        mean_mm=("prcp_mm", "mean"),
        median_mm=("prcp_mm", "median"),
        max_mm=("prcp_mm", "max")
    )
    .sort_values("median_mm", ascending=False)
    .round(2)
)
display(station_summary)

ax = station_summary["median_mm"].sort_values().plot(kind="barh", figsize=(9, 5))
ax.set(title="Median recorded daily precipitation by station", xlabel="Median (mm)", ylabel="")
plt.tight_layout()
plt.show()


## 7 · Go beyond description: test a sensible follow-up question

**New question:** At each station, on what proportion of days with an observed rainfall value did rain occur?

The denominator matters: observed days, *not all calendar days*.


In [ ]:
observed = weather.dropna(subset=["prcp_mm"]).copy()
wet_day_rate = (observed.assign(rain_occurred=observed["prcp_mm"] > 0)
               .groupby("station")["rain_occurred"]
               .agg(["size", "mean"]))
wet_day_rate["percentage_wet"] = 100 * wet_day_rate["mean"]
display(wet_day_rate[["size", "percentage_wet"]].round(1).sort_values("percentage_wet", ascending=False))


## 8 · Explain the finding, then challenge it

A chart is **not** the answer. An analyst should communicate the result and the uncertainty.

**In your own words, write four sentences:**
1. **Question:** We wanted to compare ...
2. **Evidence:** The station summaries show ... (quote values from the output).
3. **Interpretation:** Median daily rainfall is more/less appropriate than the mean because ...
4. **Limitation:** This comparison may be influenced by ... (missingness, different observed dates, location effects).

**Important:** This is observational analysis, so it cannot establish why particular stations have different values.


## 9 · Active practice (do this independently)

**A. Change the question.** Compare maximum temperatures instead of rainfall. Would you use mean/SD or median/IQR? Justify your choice after plotting.

**B. Investigate missingness.** Identify the station with the largest percentage of missing maximum-temperature records. Is it also the station with the largest percentage of missing rainfall records?

**C. Check a misconception.** Why is it wrong to replace every missing rainfall value with 0?

**D. Extend the investigation.** Plot monthly median precipitation for one station using `weather["date"].dt.month`. What pattern do you see?

**E. Tell the story.** Make a conclusion someone could understand without seeing Python.

### Self-check: can you explain these without reading?
`DataFrame` | `row` | `column` | `missing value` | `distribution` | `median` | `IQR` | `groupby` | `observational data`


In [ ]:
# Independent practice starter: monthly median precipitation at Shannon
shannon = weather.loc[weather["station"].eq("Shannon")].copy()
shannon["month"] = shannon["date"].dt.month
monthly_median = shannon.groupby("month")["prcp_mm"].median()
ax = monthly_median.plot(marker="o", figsize=(9, 4))
ax.set(title="Shannon: median daily rainfall by month (observed records)",
       xlabel="Calendar month", ylabel="Precipitation (mm)")
ax.set_xticks(range(1, 13))
plt.tight_layout()
plt.show()


## What's next?

**Week 2:** Correlation and simple/multiple regression.  
**First numbered laboratory:** Spotify Track Energy — explore relationships, fit and interpret regression models, then challenge their assumptions.

**Our rule:** The answer is never just a coefficient, a graph or a p-value. Explain what it means, what it does *not* mean, and what evidence would be needed to trust it.
